In [ ]:
# required libraries
import os, re
import pandas as pd
import numpy as np


Check for same location names

In [ ]:
import pandas as pd

df = pd.read_csv('master_df.csv')

print(df.head())

   NOLA_ID  Location State.abbreviation Plot/BagID  Block_No  Replicate_No  \
0        1  Columbia                 MO        101       1.0             1   
1        2  Columbia                 MO        202       2.0             2   
2        3  Columbia                 MO        302       2.0             3   
3        4  Columbia                 MO        401       1.0             4   
4        5  Columbia                 MO        502       2.0             5   

  Treatement_ID Residue_level Tillage Fungicide   Hybrid    Lab.code  \
0             1            HR  NoTill        NS  P1151AM  Mandy Bish   
1             1            HR  NoTill        NS  P1151AM  Mandy Bish   
2             1            HR  NoTill        NS  P1151AM  Mandy Bish   
3             1            HR  NoTill        NS  P1151AM  Mandy Bish   
4             1            HR  NoTill        NS  P1151AM  Mandy Bish   

  Notes on treatements Selected for MYCOTOXIN TESTING Planting.Date  \
0                  NaN     

In [ ]:
df["Location"].unique()

array(['Columbia', 'Albany', 'KY', 'SERF', 'MRRS', 'PPAC', 'Michigan',
       'Arlington', 'Lancaster', 'ACREE', 'NWRF'], dtype=object)

Add weather data 6 months before and after planting dates

In [ ]:
# Split 'GPS' into separate latitude and longitude columns
df[['Latitude', 'Longitude']] = df['GPS'].str.split(',', expand=True).astype(float)

df.columns

Index(['NOLA_ID', 'Location', 'State.abbreviation', 'Plot/BagID', 'Block_No',
       'Replicate_No', 'Treatement_ID', 'Residue_level', 'Tillage',
       'Fungicide', 'Hybrid', 'Lab.code', 'Notes on treatements',
       'Selected for MYCOTOXIN TESTING', 'Planting.Date', 'Harvest.date',
       'GPS', 'source', 'Latitude', 'Longitude'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np
import math
#!pip install gee_subset
import gee_subset
from gee_subset import gee_subset

In [ ]:
import pandas as pd
import numpy as np
import math
import gee_subset
from gee_subset import gee_subset

# ------------------- RH Calculation Functions -------------------

def fun_ewsis(T):
    TK = T + 273.15
    if T >= 0:
        a = [1, -6096.9385, 21.2409642, -0.02711193, 0.00001673952, 2.433502, 1]
    else:
        a = [1, -6024.5282, 29.32707, 0.010613868, -0.000013198825, -0.49382577, 1]
    return math.exp(a[1]/TK + a[2] + a[3]*TK + a[4]*TK*TK + a[5]*math.log(a[6]*TK))

def fun_alpha(T):
    if T >= 0:
        a = [0.000353624, 0.000029328363, 0.00000026168979, 0.0000000085813609]
    else:
        a = [0.000362183, 0.000026061244, 0.0000003866777,  0.0000000038268958]
    return a[0] + a[1]*T + a[2]*T*T + a[3]*T*T*T

def fun_beta(T):
    if T >= 0:
        b = [-10.7588, 0.063268134, -0.00025368934, 0.00000063405286]
    else:
        b = [-10.7604, 0.063987441, -0.00026351566, 0.0000016725084]
    return math.exp(b[0] + b[1]*T + b[2]*T*T + b[3]*T*T*T)

def fun_Pwsis(T, P):
    ewsis = fun_ewsis(T)
    f = math.exp(fun_alpha(T)*(1 - ewsis / P) + fun_beta(T)*(P / ewsis - 1))
    return ewsis * f

def fun_abs_hum(T, P, spec_hum):
    TK = T + 273.15
    return spec_hum * P / (TK * 0.2870500676)

def fun_RH(T, P, spec_hum):
    Pwsis = fun_Pwsis(T, P)
    C = 0.461522
    abs_hum = fun_abs_hum(T, P, spec_hum)
    Pw = abs_hum * (T + 273.15) * C
    return 100 * Pw / Pwsis

In [ ]:
import ee
import geemap
ee.Authenticate()
geemap.ee_initialize(project = "ee-aemshiny")

In [ ]:
pip install gee_subset

In [ ]:
import gee_subset
from gee_subset import gee_subset
import geopandas as gpd
import geemap.colormaps as cm

In [ ]:
# ----------- try to keep all the columns ---------
all_sites_data = []

for i in range(len(df)):
    row = df.iloc[i]
    latitude = row['Latitude']
    longitude = row['Longitude']
    sid = str(row['NOLA_ID'])
    location_label = str(row['Location'])

    # ✅ Ensure datetime conversion within the loop
    planting_date = pd.to_datetime(row['Planting.Date'], errors='coerce')
    date_collected = pd.to_datetime(row['Harvest.date'], errors='coerce')

    # Skip row if either date is invalid
    if pd.isna(planting_date) or pd.isna(date_collected):
        print(f"Skipping row {i} due to invalid dates.")
        continue

    # ✅ ±6 month window around Planting.Date
    start_date = (planting_date - pd.DateOffset(months=6)).strftime('%Y-%m-%d')
    end_date = (planting_date + pd.DateOffset(months=6)).strftime('%Y-%m-%d')

    # ------------------- Weather Data -------------------
    try:
        weather_df = gee_subset.gee_subset(
            product="NASA/NLDAS/FORA0125_H002",
            bands=["temperature", "wind_u", "wind_v", "total_precipitation", "specific_humidity", "pressure"],
            start_date=start_date,
            end_date=end_date,
            latitude=latitude,
            longitude=longitude,
            scale=13915
        )
        if "system:time_start" in weather_df.columns:
            weather_df.rename(columns={"system:time_start": "date"}, inplace=True)
        weather_df["date"] = pd.to_datetime(weather_df["date"])
    except Exception as e:
        print(f"Weather data error at row {i}: {e}")
        continue

    # ------------------- NDVI Data -------------------
    try:
        ndvi_df = gee_subset.gee_subset(
            product="MODIS/061/MCD43A4",
            bands=[
                "Nadir_Reflectance_Band1",
                "Nadir_Reflectance_Band2",
                "BRDF_Albedo_Band_Mandatory_Quality_Band1",
                "BRDF_Albedo_Band_Mandatory_Quality_Band2"
            ],
            start_date=start_date,
            end_date=end_date,
            latitude=latitude,
            longitude=longitude,
            scale=500
        )

        ndvi_df = ndvi_df[
            (ndvi_df["BRDF_Albedo_Band_Mandatory_Quality_Band1"] == 0) &
            (ndvi_df["BRDF_Albedo_Band_Mandatory_Quality_Band2"] == 0)
        ]

        if "NDVI" not in ndvi_df.columns:
            ndvi_df["NDVI"] = (
                ((ndvi_df["Nadir_Reflectance_Band2"] * 0.0001) - (ndvi_df["Nadir_Reflectance_Band1"] * 0.0001)) /
                ((ndvi_df["Nadir_Reflectance_Band2"] * 0.0001) + (ndvi_df["Nadir_Reflectance_Band1"] * 0.0001)).replace(0, pd.NA)
            )

        ndvi_df["date"] = pd.to_datetime(ndvi_df["date"], errors="coerce")

    except Exception as e:
        print(f"NDVI data error at row {i}: {e}")
        ndvi_df = pd.DataFrame(columns=["NDVI", "date"])  # fallback to empty df

    # ------------------- Merge & RH -------------------
    merged_df = pd.merge(weather_df, ndvi_df[["NDVI", "date"]], on="date", how="left")
    if merged_df.empty:
        print(f"No merged data at row {i}")
        continue

    merged_df["RH"] = merged_df.apply(
        lambda row: fun_RH(row["temperature"], row["pressure"], row["specific_humidity"]), axis=1
    )
    merged_df["RH"] = merged_df["RH"].clip(upper=100)

    # ✅ Add all original metadata columns from this row
    for col in df.columns:
        if col not in merged_df.columns:
            merged_df[col] = row[col]

    all_sites_data.append(merged_df)


# ------------------- Save Output -------------------
if all_sites_data:
    final_df = pd.concat(all_sites_data, ignore_index=True)
    final_df.to_csv("/content/Final2023_2024weather.csv", index=False)
else:
    print("No data to save — all rows may have failed.")


Skipping row 72 due to invalid dates.
Skipping row 73 due to invalid dates.
Skipping row 74 due to invalid dates.
Skipping row 75 due to invalid dates.
Skipping row 76 due to invalid dates.
Skipping row 77 due to invalid dates.
Skipping row 78 due to invalid dates.
Skipping row 79 due to invalid dates.
Skipping row 80 due to invalid dates.
Skipping row 81 due to invalid dates.
Skipping row 82 due to invalid dates.
Skipping row 83 due to invalid dates.
Skipping row 84 due to invalid dates.
Skipping row 85 due to invalid dates.
Skipping row 86 due to invalid dates.
Skipping row 87 due to invalid dates.
Skipping row 88 due to invalid dates.
Skipping row 89 due to invalid dates.
Skipping row 90 due to invalid dates.
Skipping row 91 due to invalid dates.
Skipping row 92 due to invalid dates.
Skipping row 93 due to invalid dates.
Skipping row 94 due to invalid dates.
not a collection
Weather data error at row 173: Image.load: Asset 'NASA/NLDAS/FORA0125_H002' is not an Image.
not a collection

In [ ]:
from google.colab import files
files.download("/content/Feb222023_2024weather.csv")


FileNotFoundError: Cannot find file: /content/Feb222023_2024weather.csv

In [ ]:
# --------------------find daily averages----------------------------
import pandas as pd

# Ensure datetime columns are correctly parsed
final_df['date'] = pd.to_datetime(final_df['date'], errors='coerce')
final_df['Planting.Date'] = pd.to_datetime(final_df['Planting.Date'], errors='coerce')
final_df['Harvest.date'] = pd.to_datetime(final_df['Harvest.date'], errors='coerce')

# Extract just the date (without time)
final_df['day'] = final_df['date'].dt.date

# Define grouping columns (metadata to preserve)
group_cols = ['day', 'NOLA_ID', 'Location', 'latitude', 'longitude', 'Planting.Date', 'Harvest.date']

# Build aggregation dictionary for all other columns
agg_dict = {
    col: (lambda x: x.mean() if pd.api.types.is_numeric_dtype(x) else x.iloc[0])
    for col in final_df.columns if col not in group_cols
}

# Perform grouping and aggregation
daily_avg = (
    final_df
    .groupby(group_cols)
    .agg(agg_dict)
    .reset_index()
)

# Add derived variable: days from planting
daily_avg['days_from_planting'] = (pd.to_datetime(daily_avg['day']) - daily_avg['Planting.Date']).dt.days

# Save output
daily_avg.to_csv("/content/2023_2024weather_daily.csv", index=False)


In [ ]:
df.isna().sum()


,0
Hybrid,0
NOLA_ID,0
Selected.for.mycotoxin.testing,0
Year,0
Lab.code,0
...,...
VT2Pro,0
GPS,0
source,0
Latitude,0


In [ ]:
df.shape

(320, 64)